In [1]:
import polars as pl
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error, root_mean_squared_error

from catboost import CatBoostRegressor

import optuna

import nbformat

import warnings

warnings.filterwarnings('ignore')

/Users/egor/VS_GIT_repositories/BYTE/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet('/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_cleaned.parquet')
df.shape

(30006, 22)

In [3]:
df.head()

,offer_id,price,price_numeric,old_price,area,rooms,floor,price_per_m2,metro,metro_time,...,main_image,photo_count,badges,publish_date,url,title,description,image_urls,self_floor,max_floor
0,7035113340557126091,7 500 000 ₽,7500000.0,NaN,17.7,студия,9 этаж из 16,None,Калитники,9.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,None,https://realty.yandex.ru/offer/703511334055712...,апартаменты-студия,Номер лота: 99696. Панорамный вид из больших о...,https://avatars.mds.yandex.net/get-realty-offe...,9.0,16.0
1,7035113340416809607,7 500 000 ₽,7500000.0,NaN,17.0,студия,2 этаж из 2,None,Соколиная гора,8.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/703511334041680...,апартаменты-студия,Номер лота: 87440. Продается студия с дизайнер...,https://avatars.mds.yandex.net/get-realty-offe...,2.0,2.0
2,7053956964805047621,12 200 000 ₽,12200000.0,NaN,17.9,студия,2 этаж из 48,None,Тушинская,10.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,3 квартал 2027,https://realty.yandex.ru/offer/705395696480504...,квартира-студия,"Арт. 119802099 Студия 17,9 м в CITYZEN Урбан-б...",https://avatars.mds.yandex.net/get-realty-offe...,2.0,48.0
3,7053956914445503237,7 300 000 ₽,7300000.0,NaN,15.7,студия,5 этаж из 5,None,Бутырская,17.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/705395691444550...,квартира-студия,Арт. 134010491 СПЕЦИАЛЬНО для наших клиентов с...,https://avatars.mds.yandex.net/get-realty-offe...,5.0,5.0
4,3699730400767130013,10 802 031 ₽,10802031.0,NaN,14.1,студия,5 этаж из 16,None,Коммунарка,14.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,2 квартал 2026,https://realty.yandex.ru/offer/369973040076713...,квартира-студия,Строим кварталы для жизни с заботой о будущем....,https://avatars.mds.yandex.net/get-realty-offe...,5.0,16.0


In [10]:
def objective(trial):
    params = {
        "iterations": 5000,
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.5, log=True),
        "depth": trial.suggest_int("depth", 4, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-9, 20.0, log=True),
        "border_count": trial.suggest_int("border_count", 1, 255),
        "grow_policy": trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
        "rsm": trial.suggest_float("rsm", 0.1, 1.0), 
        
        # --- Продвинутые параметры ---
        
        # Частота обновления весов признаков (ускоряет обучение на больших данных)
        "feature_border_type": trial.suggest_categorical("feature_border_type", 
            ["Median", "Uniform", "UniformAndQuantiles", "MaxLogSum", "MinEntropy"]),
        
        # Параметры для работы с категориальными фичами (CTR)
        "max_ctr_complexity": trial.suggest_int("max_ctr_complexity", 1, 4),
        "one_hot_max_size": trial.suggest_int("one_hot_max_size", 0, 25), # 0 = отключено
        
        # Сглаживание при расчете средних для категорий
        "model_shrink_rate": trial.suggest_float("model_shrink_rate", 0, 1.0),
        "model_shrink_mode": trial.suggest_categorical("model_shrink_mode", ["Constant", "Decreasing"]),

        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "od_type": "Iter",
        "od_wait": 50,
        "verbose": False,
        "random_state": 42
    }

    # Логика зависимых параметров
    bootstrap_type = trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS", "No"])
    params["bootstrap_type"] = bootstrap_type
    
    if bootstrap_type == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10)
    elif bootstrap_type in ["Bernoulli", "MVS"]:
        params["subsample"] = trial.suggest_float("subsample", 0.1, 1)

    if params["grow_policy"] != "SymmetricTree":
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 1, 100)
    
    if params["grow_policy"] == "Lossguide":
        params["max_leaves"] = trial.suggest_int("max_leaves", 16, 128)

    reg = CatBoostRegressor(**params,
    cat_features=['metro', 'title', 'author'],
    #text_features=['description', 'address', 'metro'],
    )

    reg.fit(X_train, Y_train, eval_set=[(X_val, Y_val)], early_stopping_rounds=100)

    return root_mean_squared_error(Y_val, reg.predict(X_val))


In [11]:
X = df[['area', 'metro_time', 'photo_count', 
'self_floor', 'max_floor', 'metro', 'title', 
'author', 
#'description', 'address'
]]

X['metro'] = X['metro'].apply(lambda x: str(x))
X['title'] = X['title'].apply(lambda x: str(x))
X['author'] = X['author'].apply(lambda x: str(x))
#X['description'] = X['description'].apply(lambda x: str(x))
#X['address'] = X['address'].apply(lambda x: str(x))


Y = df['price_numeric']

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, shuffle=True)

In [12]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print(f"Лучший RMSE: {study.best_value}")
print(f"Параметры: {study.best_params}")

[I 2026-04-09 02:27:55,102] A new study created in memory with name: no-name-777528f3-d4a8-451e-b102-48c1f4072f5f
[I 2026-04-09 02:27:56,514] Trial 0 finished with value: 8480742.258977626 and parameters: {'learning_rate': 0.057935181117343755, 'depth': 10, 'l2_leaf_reg': 0.0002208071227575817, 'random_strength': 1.0689260643001154e-06, 'border_count': 10, 'grow_policy': 'Lossguide', 'min_data_in_leaf': 36, 'rsm': 0.13753390526748827, 'feature_border_type': 'UniformAndQuantiles', 'max_ctr_complexity': 4, 'one_hot_max_size': 4, 'model_shrink_rate': 0.2532859733310483, 'model_shrink_mode': 'Constant', 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.38510199849899784, 'max_leaves': 35}. Best is trial 0 with value: 8480742.258977626.
[I 2026-04-09 02:28:37,028] Trial 1 finished with value: 8429505.467112364 and parameters: {'learning_rate': 0.0004891833919683355, 'depth': 6, 'l2_leaf_reg': 4.727771383776454e-08, 'random_strength': 11.730208729712023, 'border_count': 22, 'grow_policy

Лучший RMSE: 4019548.183199788
Параметры: {'learning_rate': 0.07240029457505619, 'depth': 8, 'l2_leaf_reg': 0.47154458576232194, 'random_strength': 14.618825667749789, 'border_count': 253, 'grow_policy': 'SymmetricTree', 'min_data_in_leaf': 60, 'rsm': 0.9964534252040771, 'feature_border_type': 'Median', 'max_ctr_complexity': 4, 'one_hot_max_size': 22, 'model_shrink_rate': 0.38764528904394, 'model_shrink_mode': 'Decreasing', 'bootstrap_type': 'No'}


In [19]:
optuna.visualization.plot_optimization_history(study)

In [18]:
best_params = {
'learning_rate': 0.07240029457505619,
 'depth': 8,
 'l2_leaf_reg': 0.47154458576232194,
 'random_strength': 14.618825667749789,
 'border_count': 253,
 'grow_policy': 'SymmetricTree',
 'min_data_in_leaf': 60,
 'rsm': 0.9964534252040771,
 'feature_border_type': 'Median',
 'max_ctr_complexity': 4,
 'one_hot_max_size': 22,
 'model_shrink_rate': 0.38764528904394,
 'model_shrink_mode': 'Decreasing',
 'bootstrap_type': 'No'}

model = CatBoostRegressor(**best_params,
cat_features=['metro', 'title', 'author'],
#text_features=['description', 'address', 'metro'],
)

model.fit(X_train, Y_train, eval_set=[(X_val, Y_val)], early_stopping_rounds=100)

0:	learn: 10325985.2972859	test: 10646286.1087392	best: 10646286.1087392 (0)	total: 34.5ms	remaining: 34.5s
1:	learn: 11369200.0766041	test: 11678212.3342415	best: 10646286.1087392 (0)	total: 51.7ms	remaining: 25.8s
2:	learn: 11862933.0872440	test: 12202256.9540174	best: 10646286.1087392 (0)	total: 65.9ms	remaining: 21.9s
3:	learn: 12029871.1167324	test: 12372626.8245634	best: 10646286.1087392 (0)	total: 76.8ms	remaining: 19.1s
4:	learn: 12021848.9860728	test: 12340300.8359563	best: 10646286.1087392 (0)	total: 84.2ms	remaining: 16.8s
5:	learn: 11929841.4547787	test: 12246586.3015865	best: 10646286.1087392 (0)	total: 97.4ms	remaining: 16.1s
6:	learn: 11770845.3573058	test: 12088202.5613780	best: 10646286.1087392 (0)	total: 108ms	remaining: 15.3s
7:	learn: 11613501.2174612	test: 11931819.3368521	best: 10646286.1087392 (0)	total: 115ms	remaining: 14.2s
8:	learn: 11380859.3675027	test: 11703289.4848905	best: 10646286.1087392 (0)	total: 121ms	remaining: 13.3s
9:	learn: 11167991.0304307	test

CatBoostRegressor(bootstrap_type='No', border_count=253, cat_features=['metro', 'title', 'author'], depth=8, feature_border_type='Median', grow_policy='SymmetricTree', l2_leaf_reg=0.47154458576232194, learning_rate=0.07240029457505619, loss_function='RMSE', max_ctr_complexity=4, min_data_in_leaf=60, model_shrink_mode='Decreasing', model_shrink_rate=0.38764528904394, one_hot_max_size=22, random_strength=14.618825667749789, rsm=0.9964534252040771)

In [20]:
def eval_with_metrics(model, X, Y):
    
    preds = model.predict(X)

    print(f"R^2: {r2_score(Y, preds)} \n"
          f"MAE: {mean_absolute_error(Y, preds)} \n"
          f"MAPE: {mean_absolute_percentage_error(Y, preds)} \n"
          f"RMSE: {root_mean_squared_error(Y, preds)} \n")
    
eval_with_metrics(model, X_val, Y_val)

R^2: 0.8568431779016246 
MAE: 1901654.9099309172 
MAPE: 0.14206282525526556 
RMSE: 4164723.1437319177 

